# Lab 08 - Convolutional Neural Networks: LeNet-5 and AlexNet on MNIST

**Course:** Computer Vision  |  **Framework:** PyTorch  |  **Dataset:** MNIST (auto-downloaded by torchvision)

**How to run on Google Colab:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
AlexNet is large (about 56.9M parameters, 227x227 input), so a GPU is strongly recommended.

**Notebook contents (matches the Lab Tasks table):**
1. Architecture implementation
2. Dimension verification
3. Parameter analysis (manual + PyTorch)
4. Training and testing
5. Feature map interpretation
6. Critical comparison
7. Interpretation questions (Section 5.1) and submission items

In [ ]:
# Install (Colab already has these, this is only a safety net)
!pip -q install torch torchvision matplotlib pandas

In [ ]:
import os, time, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cpu":
    print("WARNING: No GPU found. AlexNet will be very slow. Use Runtime > Change runtime type > T4 GPU.")

## Part A - Manual Calculations (Sections 1.1 and 1.2)

Helper functions implementing the formulas from the lab manual:

* Output size: `H_out = floor((H + 2P - K) / S) + 1`
* Conv parameters: `C_out x [(C_in / G) x K_h x K_w + 1]`
* FC parameters: `(in x out) + out`

In [ ]:
def conv_out(size, K, S=1, P=0):
    return math.floor((size + 2*P - K) / S) + 1

def conv_params(c_in, c_out, k, groups=1):
    return c_out * ((c_in // groups) * k * k + 1)

def fc_params(n_in, n_out):
    return n_in * n_out + n_out

# ---------- Worked example from the manual ----------
print("Conv1 AlexNet output size:", conv_out(227, 11, S=4, P=0), "-> 96 x 55 x 55")

# ---------- LeNet-5 manual dimension + parameter table ----------
s = 32
rows_lenet = []
s = conv_out(s, 5);           rows_lenet.append(("C1 Conv 5x5, 6",   f"6 x {s} x {s}",   conv_params(1, 6, 5)))
s = conv_out(s, 2, S=2);      rows_lenet.append(("S2 AvgPool 2x2",   f"6 x {s} x {s}",   0))
s = conv_out(s, 5);           rows_lenet.append(("C3 Conv 5x5, 16",  f"16 x {s} x {s}",  conv_params(6, 16, 5)))
s = conv_out(s, 2, S=2);      rows_lenet.append(("S4 AvgPool 2x2",   f"16 x {s} x {s}",  0))
s = conv_out(s, 5);           rows_lenet.append(("C5 Conv 5x5, 120", f"120 x {s} x {s}", conv_params(16, 120, 5)))
rows_lenet.append(("F6 Linear 120->84", "84", fc_params(120, 84)))
rows_lenet.append(("Output Linear 84->10", "10", fc_params(84, 10)))
df_lenet_manual = pd.DataFrame(rows_lenet, columns=["Layer", "Output (C x H x W)", "Params (manual)"])
display(df_lenet_manual)
print("LeNet-5 manual total:", f"{df_lenet_manual['Params (manual)'].sum():,}")

# ---------- AlexNet manual dimension + parameter table ----------
s = 227
rows_alex = []
s = conv_out(s, 11, S=4);     rows_alex.append(("Conv1 11x11, 96, S=4",        f"96 x {s} x {s}",  conv_params(1, 96, 11)))
s = conv_out(s, 3, S=2);      rows_alex.append(("Pool1 3x3, S=2",              f"96 x {s} x {s}",  0))
s = conv_out(s, 5, S=1, P=2); rows_alex.append(("Conv2 5x5, 256, P=2, G=2",    f"256 x {s} x {s}", conv_params(96, 256, 5, groups=2)))
s = conv_out(s, 3, S=2);      rows_alex.append(("Pool2 3x3, S=2",              f"256 x {s} x {s}", 0))
s = conv_out(s, 3, S=1, P=1); rows_alex.append(("Conv3 3x3, 384, P=1",         f"384 x {s} x {s}", conv_params(256, 384, 3)))
s = conv_out(s, 3, S=1, P=1); rows_alex.append(("Conv4 3x3, 384, P=1, G=2",    f"384 x {s} x {s}", conv_params(384, 384, 3, groups=2)))
s = conv_out(s, 3, S=1, P=1); rows_alex.append(("Conv5 3x3, 256, P=1, G=2",    f"256 x {s} x {s}", conv_params(384, 256, 3, groups=2)))
s = conv_out(s, 3, S=2);      rows_alex.append(("Pool3 3x3, S=2",              f"256 x {s} x {s}", 0))
flat = 256 * s * s
rows_alex.append(("Flatten", f"{flat:,} values", 0))
rows_alex.append(("FC6 Linear 9216->4096", "4096", fc_params(flat, 4096)))
rows_alex.append(("FC7 Linear 4096->4096", "4096", fc_params(4096, 4096)))
rows_alex.append(("FC8 Linear 4096->10",   "10",   fc_params(4096, 10)))
df_alex_manual = pd.DataFrame(rows_alex, columns=["Layer", "Output (C x H x W)", "Params (manual)"])
display(df_alex_manual)
print("AlexNet-MNIST manual total:", f"{df_alex_manual['Params (manual)'].sum():,}")

### Hand calculations for Task 3 (Conv1, Conv2, FC6, FC7, FC8)

| Layer | Calculation | Result |
|---|---|---|
| Conv1 | 96 x [(1 x 11 x 11) + 1] = 96 x 122 | **11,712** |
| Conv2 (G=2) | 256 x [(96/2 x 5 x 5) + 1] = 256 x [(48 x 25) + 1] = 256 x 1,201 | **307,456** |
| FC6 | (9,216 x 4,096) + 4,096 | **37,752,832** |
| FC7 | (4,096 x 4,096) + 4,096 | **16,781,312** |
| FC8 (MNIST) | (4,096 x 10) + 10 | **40,970** |

Extra (LeNet-5): C1 = 6 x [(1x5x5)+1] = 156, C3 = 16 x [(6x5x5)+1] = 2,416, C5 = 120 x [(16x5x5)+1] = 48,120,
F6 = (120x84)+84 = 10,164, Output = (84x10)+10 = 850. **Total = 61,706.**

## Part B - Task 1: Architecture Implementation

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),            # 32 -> 28
            nn.Tanh(), nn.AvgPool2d(kernel_size=2, stride=2),   # 28 -> 14
            nn.Conv2d(6, 16, kernel_size=5),           # 14 -> 10
            nn.Tanh(), nn.AvgPool2d(kernel_size=2, stride=2),   # 10 -> 5
            nn.Conv2d(16, 120, kernel_size=5),         # 5 -> 1
            nn.Tanh()
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(120, 84), nn.Tanh(), nn.Linear(84, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)   # return logits


class AlexNetMNIST(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 96, kernel_size=11, stride=4),               # 227 -> 55
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),                    # 55 -> 27
            nn.Conv2d(96, 256, kernel_size=5, padding=2, groups=2),   # 27 -> 27
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),                    # 27 -> 13
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1, groups=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1, groups=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)                     # 13 -> 6
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5), nn.Linear(256 * 6 * 6, 4096), nn.ReLU(inplace=True),
            nn.Dropout(p=0.5), nn.Linear(4096, 4096), nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)   # return logits

lenet = LeNet5().to(device)
alexnet = AlexNetMNIST().to(device)
print(lenet)
print()
print(alexnet)

## Data Loaders (MNIST is downloaded automatically)

In [ ]:
def make_mnist_loaders(image_size, batch_size, pad_to_32=False):
    transform_ops = []
    if pad_to_32:
        transform_ops.append(transforms.Pad(2))                       # 28 -> 32 (no stretching)
    else:
        transform_ops.append(transforms.Resize((image_size, image_size)))
    transform_ops += [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
    transform = transforms.Compose(transform_ops)

    train_set = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
    test_set  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    pin = (device.type == "cuda")
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=pin)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin)
    return train_loader, test_loader

# Separate loaders because the networks use different input sizes
lenet_train, lenet_test = make_mnist_loaders(image_size=32,  batch_size=64, pad_to_32=True)
alex_train,  alex_test  = make_mnist_loaders(image_size=227, batch_size=16, pad_to_32=False)
print("Train samples:", len(lenet_train.dataset), "| Test samples:", len(lenet_test.dataset))

## Part C - Task 2: Dimension Verification

Forward hooks record the output shape after every layer, so we can verify every convolution and pooling stage and the 6 x 6 x 256 tensor before flattening.

In [ ]:
def layer_shapes(model, x):
    records, hooks = [], []
    def make_hook(name):
        def hook(module, inp, out):
            records.append((name, type(module).__name__, tuple(out.shape)))
        return hook
    for name, m in model.named_modules():
        if len(list(m.children())) == 0:      # leaf layers only
            hooks.append(m.register_forward_hook(make_hook(name)))
    model.eval()
    with torch.no_grad():
        model(x.to(device))
    for h in hooks: h.remove()
    return pd.DataFrame(records, columns=["Layer name", "Layer type", "Output shape (B, C, H, W)"])

x_lenet, y_lenet = next(iter(lenet_train))
x_alex,  y_alex  = next(iter(alex_train))
print("LeNet input :", tuple(x_lenet.shape))
print("LeNet output:", tuple(lenet(x_lenet.to(device)).shape))
print("AlexNet input :", tuple(x_alex.shape))
print("AlexNet output:", tuple(alexnet(x_alex.to(device)).shape))

print("\n=== LeNet-5 layer-by-layer output shapes ===")
display(layer_shapes(lenet, x_lenet))
print("\n=== AlexNet-MNIST layer-by-layer output shapes ===")
df_alex_shapes = layer_shapes(alexnet, x_alex)
display(df_alex_shapes)

# Explicit check of the tensor before flattening
with torch.no_grad():
    alexnet.eval()
    feat = alexnet.features(x_alex.to(device))
print("\nAlexNet tensor before flatten:", tuple(feat.shape[1:]), "-> flattened:", feat[0].numel())
assert tuple(feat.shape[1:]) == (256, 6, 6) and feat[0].numel() == 9216
print("Verified: 256 x 6 x 6 = 9,216")

## Part D - Task 3: Parameter Analysis (manual vs PyTorch)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

lenet_total  = count_parameters(lenet)
alex_total   = count_parameters(alexnet)
print("LeNet-5 parameters       :", f"{lenet_total:,}",  "(expected 61,706)")
print("AlexNet-MNIST parameters :", f"{alex_total:,}", "(expected 56,885,962)")
assert lenet_total == 61706 and alex_total == 56885962

# Per-layer comparison: manual formula vs PyTorch
def per_layer_params(model):
    out = []
    for name, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            out.append((name, type(m).__name__, sum(p.numel() for p in m.parameters())))
    return out

alex_manual = [r[2] for r in rows_alex if r[2] > 0]
alex_torch = per_layer_params(alexnet)
df_cmp = pd.DataFrame({
    "PyTorch layer": [f"{n} ({t})" for n, t, _ in alex_torch],
    "PyTorch params": [p for _, _, p in alex_torch],
    "Manual params": alex_manual,
})
df_cmp["Match"] = df_cmp["PyTorch params"] == df_cmp["Manual params"]
display(df_cmp)

# Effect of groups=2 (Question 7)
no_group = conv_params(96, 256, 5, groups=1)
with_group = conv_params(96, 256, 5, groups=2)
print(f"\nConv2 without groups: {no_group:,} | with groups=2: {with_group:,} | saving: {no_group - with_group:,}")

fc_share = (fc_params(9216,4096) + fc_params(4096,4096) + fc_params(4096,10)) / alex_total * 100
print(f"FC layers hold {fc_share:.1f}% of all AlexNet-MNIST parameters")

## Part E - Task 4: Training and Testing

Training uses `CrossEntropyLoss` on raw logits (no Softmax) and the Adam optimizer. The training function also records time per epoch.

In [ ]:
def run_training_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()                      # disables dropout
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

def fit_model(model, train_loader, test_loader, epochs=5, learning_rate=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    history, epoch_times = [], []
    for epoch in range(epochs):
        t0 = time.time()
        train_loss, train_acc = run_training_epoch(model, train_loader, criterion, optimizer)
        test_loss, test_acc = evaluate(model, test_loader, criterion)
        if device.type == "cuda": torch.cuda.synchronize()
        dt = time.time() - t0
        history.append((train_loss, train_acc, test_loss, test_acc))
        epoch_times.append(dt)
        print(f"Epoch {epoch+1}/{epochs} | Train loss: {train_loss:.4f} | Train acc: {train_acc:.2f}% | "
              f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.2f}% | Time: {dt:.1f}s")
    return history, epoch_times

In [ ]:
# ---------------- Train LeNet-5 ----------------
LENET_EPOCHS = 5
print("Training LeNet-5 ...")
lenet_history, lenet_times = fit_model(lenet, lenet_train, lenet_test, epochs=LENET_EPOCHS, learning_rate=1e-3)
torch.save(lenet.state_dict(), "lenet5_mnist.pth")
print("Saved lenet5_mnist.pth")

In [ ]:
# ---------------- Train AlexNet-MNIST ----------------
# Needs a GPU. If out-of-memory: reduce batch size to 8 (re-create alex_train/alex_test) and restart runtime.
# If too slow: reduce ALEX_EPOCHS to 1 or 2.
ALEX_EPOCHS = 3
print("Training AlexNet-MNIST ...")
alexnet_history, alexnet_times = fit_model(alexnet, alex_train, alex_test, epochs=ALEX_EPOCHS, learning_rate=1e-4)
torch.save(alexnet.state_dict(), "alexnet_mnist.pth")
print("Saved alexnet_mnist.pth")

In [ ]:
# Learning curves
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row, (name, hist) in enumerate([("LeNet-5", lenet_history), ("AlexNet-MNIST", alexnet_history)]):
    ep = range(1, len(hist) + 1)
    axes[row, 0].plot(ep, [h[0] for h in hist], "o-", label="Train loss")
    axes[row, 0].plot(ep, [h[2] for h in hist], "s--", label="Test loss")
    axes[row, 0].set_title(f"{name} - Loss"); axes[row, 0].set_xlabel("Epoch"); axes[row, 0].set_ylabel("Cross-entropy loss"); axes[row, 0].legend(); axes[row, 0].grid(alpha=.3)
    axes[row, 1].plot(ep, [h[1] for h in hist], "o-", label="Train acc")
    axes[row, 1].plot(ep, [h[3] for h in hist], "s--", label="Test acc")
    axes[row, 1].set_title(f"{name} - Accuracy"); axes[row, 1].set_xlabel("Epoch"); axes[row, 1].set_ylabel("Accuracy (%)"); axes[row, 1].legend(); axes[row, 1].grid(alpha=.3)
    axes[row, 0].set_xticks(list(ep)); axes[row, 1].set_xticks(list(ep))
plt.tight_layout(); plt.show()

## Comparison Table (Section 4.9, measured values)

In [ ]:
def file_mb(path): return os.path.getsize(path) / (1024 * 1024)

# Final accuracy on the full train split with eval() mode (dropout off), plus test accuracy
crit = nn.CrossEntropyLoss()
lenet_train_eval_loader = DataLoader(lenet_train.dataset, batch_size=256, shuffle=False, num_workers=2)
alex_train_eval_loader  = DataLoader(alex_train.dataset,  batch_size=64,  shuffle=False, num_workers=2)
l_tr_loss, l_tr_acc = evaluate(lenet,   lenet_train_eval_loader, crit)
l_te_loss, l_te_acc = evaluate(lenet,   lenet_test,              crit)
a_tr_loss, a_tr_acc = evaluate(alexnet, alex_train_eval_loader,  crit)
a_te_loss, a_te_acc = evaluate(alexnet, alex_test,               crit)

comparison = pd.DataFrame({
    "Measure": ["Input image size", "Trainable parameters", "Epochs completed",
                "Final training accuracy (%)", "Final test accuracy (%)",
                "Final training loss", "Final test loss",
                "Training time per epoch (s, avg)", "Model file size (MB)"],
    "LeNet-5": ["32 x 32", f"{lenet_total:,}", LENET_EPOCHS, f"{l_tr_acc:.2f}", f"{l_te_acc:.2f}",
                f"{l_tr_loss:.4f}", f"{l_te_loss:.4f}", f"{np.mean(lenet_times):.1f}", f"{file_mb('lenet5_mnist.pth'):.2f}"],
    "AlexNet-MNIST": ["227 x 227", f"{alex_total:,}", ALEX_EPOCHS, f"{a_tr_acc:.2f}", f"{a_te_acc:.2f}",
                f"{a_tr_loss:.4f}", f"{a_te_loss:.4f}", f"{np.mean(alexnet_times):.1f}", f"{file_mb('alexnet_mnist.pth'):.2f}"],
})
display(comparison.set_index("Measure"))

## Part F - Error Analysis (selected misclassified test images)

In [ ]:
@torch.no_grad()
def collect_errors(model, loader, n=8):
    model.eval()
    errs = []
    for images, labels in loader:
        preds = model(images.to(device)).argmax(1).cpu()
        for img, y, p in zip(images, labels, preds):
            if y != p:
                errs.append((img, int(y), int(p)))
                if len(errs) >= n: return errs
    return errs

@torch.no_grad()
def confusion(model, loader):
    model.eval()
    cm = np.zeros((10, 10), dtype=int)
    for images, labels in loader:
        preds = model(images.to(device)).argmax(1).cpu()
        for y, p in zip(labels, preds): cm[int(y), int(p)] += 1
    return cm

for name, model, loader in [("LeNet-5", lenet, lenet_test), ("AlexNet-MNIST", alexnet, alex_test)]:
    errs = collect_errors(model, loader, n=8)
    fig, axes = plt.subplots(1, len(errs), figsize=(2 * max(len(errs), 1), 2.6))
    axes = np.atleast_1d(axes)
    for ax, (img, y, p) in zip(axes, errs):
        ax.imshow(img[0] * 0.3081 + 0.1307, cmap="gray")
        ax.set_title(f"True {y}\nPred {p}", fontsize=9); ax.axis("off")
    fig.suptitle(f"{name}: first misclassified test images"); plt.tight_layout(); plt.show()

cm_l = confusion(lenet, lenet_test); cm_a = confusion(alexnet, alex_test)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cm, name in zip(axes, [cm_l, cm_a], ["LeNet-5", "AlexNet-MNIST"]):
    im = ax.imshow(cm, cmap="Blues"); ax.set_title(f"{name} confusion matrix")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_xticks(range(10)); ax.set_yticks(range(10))
    for i in range(10):
        for j in range(10):
            if cm[i, j] > 0 and i != j: ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="red")
plt.tight_layout(); plt.show()

## Part G - Task 5: Feature Map Visualization (first convolution of AlexNet)

A forward hook on `alexnet.features[0]` captures the output of Conv1 for one normalized MNIST test image. We show the input and **8** feature maps.

In [ ]:
saved_activations = {}
def capture_first_conv(module, inputs, output):
    saved_activations["conv1"] = output.detach().cpu()

hook = alexnet.features[0].register_forward_hook(capture_first_conv)
alexnet.eval()
sample, label = alex_test.dataset[0]
with torch.no_grad():
    _ = alexnet(sample.unsqueeze(0).to(device))
hook.remove()

maps = saved_activations["conv1"][0]          # (96, 55, 55)
print("Captured Conv1 feature maps:", tuple(maps.shape), "| true label:", label)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes[0, 0].imshow(sample[0] * 0.3081 + 0.1307, cmap="gray")
axes[0, 0].set_title(f"Input digit (label {label}), 227x227"); axes[0, 0].axis("off")
for i, ax in enumerate(axes.flat[1:]):
    ax.imshow(maps[i], cmap="gray")
    ax.set_title(f"Filter {i}"); ax.axis("off")
plt.tight_layout(); plt.show()

# Simple quantitative support for the interpretation
stats = pd.DataFrame({"Filter": range(8),
                      "Mean": [float(maps[i].mean()) for i in range(8)],
                      "Std":  [float(maps[i].std())  for i in range(8)],
                      "Min":  [float(maps[i].min())  for i in range(8)],
                      "Max":  [float(maps[i].max())  for i in range(8)]})
display(stats.round(3))

### Feature map interpretation

* The input digit is a 227 x 227 image; each Conv1 map is 55 x 55, because the 11 x 11 filters with stride 4 sample the image coarsely, so the digit appears as a smaller, blockier version.
* Different filters respond differently to the **same** digit: some maps are bright mainly along the stroke (they respond to stroke or intensity), some highlight stroke boundaries or one orientation of edges (for example vertical or diagonal parts), and some are weak or nearly uniform (low standard deviation in the table above), meaning that filter is not strongly activated by this digit.
* Maps keep the digit's overall spatial layout (position of the loop or stem), since convolution is spatially shared and preserves where a pattern occurs.
* These are learned responses. Not every filter maps to one human-readable feature, and with only 3 epochs on MNIST some filters stay close to their random initialisation, so their maps look like noisy blurred copies of the digit.
* Check the figure and the statistics table from your own run and adjust the sentences above to match what you actually see, since results vary with the random initialisation.

## Part H - Section 5.1: Questions for AlexNet Feature Map Interpretation

**1. Why does the spatial resolution decrease from 227 x 227 to 55 x 55 after Conv1?**
Conv1 uses a large 11 x 11 kernel with **stride 4** and no padding. Using the formula, floor((227 - 11) / 4) + 1 = 55. The stride of 4 means the filter jumps 4 pixels at a time, so only about every fourth position is sampled. This quickly reduces spatial size and computation, while the 96 filters keep rich information in the channel dimension.

**2. Why do the number of channels increase from 96 in Conv1 to 256 in Conv2 and 384 in Conv3?**
Spatial size shrinks as we go deeper, but the network needs to represent more and more complex patterns. Early layers detect simple edges and colour or intensity blobs; later layers combine them into textures, parts and shapes, and there are many more possible combinations, so more filters (channels) are needed. Increasing depth while reducing resolution is the standard CNN pattern.

**3. What is the purpose of padding = 2 in Conv2 and padding = 1 in Conv3, Conv4, and Conv5?**
It is "same" padding: it keeps the spatial size unchanged. For Conv2 (5 x 5 kernel), P = (K - 1) / 2 = 2, so floor((27 + 4 - 5) / 1) + 1 = 27. For the 3 x 3 convolutions, P = 1 gives floor((13 + 2 - 3) / 1) + 1 = 13. Padding also lets border pixels take part in as many convolutions as central pixels, so border information is not lost and size reduction is left to the pooling layers.

**4. Why does the output remain 13 x 13 through Conv3, Conv4, and Conv5?**
All three use a 3 x 3 kernel, stride 1 and padding 1, so H_out = floor((13 + 2 x 1 - 3) / 1) + 1 = 13. There is no pooling between them, so the spatial size stays 13 x 13 and only the number of channels changes (256 -> 384 -> 384 -> 256).

**5. Why does Pool3 produce 6 x 6 x 256, and why does flattening give 9,216 values?**
Pool3 is a 3 x 3 max pool with stride 2 on a 13 x 13 map: floor((13 - 3) / 2) + 1 = 6. Pooling does not change the channel count, so the output is 256 x 6 x 6. Flattening multiplies them: 256 x 6 x 6 = 9,216 values, which is the input size of FC6.

**6. Which layers contain the largest number of trainable parameters? Explain why.**
FC6 (37,752,832) is largest, then FC7 (16,781,312). Together with FC8 they hold about 96% of the 56,885,962 parameters. Fully connected layers connect every input unit to every output unit and have no weight sharing: FC6 alone has 9,216 x 4,096 weights. Among convolutional layers, Conv3 (885,120) is largest, but convolution filters are small and shared across all spatial positions, so they need far fewer parameters.

**7. What is the effect of using groups = 2 in Conv2, Conv4, and Conv5?**
Each filter is connected to only half of the input channels (C_in / 2), so the number of weights and multiply-adds in those layers is halved. For Conv2 it is 307,456 instead of 614,656 without groups. The channels are split into two independent streams (originally to spread the network across two GPUs), but this restricts cross-channel mixing inside those layers, so each group learns features from only half of the previous channels.

**8. Why can AlexNet have many more parameters than LeNet-5 even though both classify ten MNIST classes?**
The number of parameters depends on the architecture, not on the number of classes. AlexNet is deeper, uses many more filters (96 to 384 channels), and processes a much larger 227 x 227 input, which gives a 6 x 6 x 256 = 9,216 feature vector feeding two 4,096-unit dense layers. LeNet-5 works on 32 x 32 input and compresses to only 120 features before a small 84-unit layer. The 56.9M vs 61.7K parameters (about 920 times larger) are far more than the simple 10-class MNIST task needs, and the network stays that size because it was designed for 1,000-class ImageNet.

## Part I - Task 6: Critical Comparison (auto-generated from your measured results)

In [ ]:
gap_l = l_tr_acc - l_te_acc
gap_a = a_tr_acc - a_te_acc
size_ratio = alex_total / lenet_total
time_ratio = np.mean(alexnet_times) / np.mean(lenet_times)
mb_ratio = file_mb('alexnet_mnist.pth') / file_mb('lenet5_mnist.pth')

print(f"Model size      : AlexNet has {size_ratio:,.0f}x more parameters than LeNet-5 "
      f"({file_mb('alexnet_mnist.pth'):.1f} MB vs {file_mb('lenet5_mnist.pth'):.2f} MB file, {mb_ratio:,.0f}x).")
print(f"Training time   : AlexNet epoch takes {np.mean(alexnet_times):.1f}s vs {np.mean(lenet_times):.1f}s for LeNet-5 ({time_ratio:.1f}x slower).")
print(f"Test accuracy   : LeNet-5 {l_te_acc:.2f}% vs AlexNet {a_te_acc:.2f}%  (difference {a_te_acc - l_te_acc:+.2f} points).")
print(f"Generalization  : train-test accuracy gap  LeNet-5 {gap_l:+.2f} | AlexNet {gap_a:+.2f} points;")
print(f"                  train-test loss gap      LeNet-5 {l_te_loss - l_tr_loss:+.4f} | AlexNet {a_te_loss - a_tr_loss:+.4f}.")

### Discussion

* **Model size:** AlexNet-MNIST has about 56.9 million parameters versus 61.7 thousand for LeNet-5 (roughly 920 times more), and its saved `.pth` file is correspondingly much larger. Most of this comes from FC6 and FC7.
* **Training time and computation:** AlexNet works on 227 x 227 inputs (about 52 times more pixels than 32 x 32) with large feature maps and dense layers, so each epoch is much slower and needs far more GPU memory. LeNet-5 trains in seconds even on a CPU.
* **Test accuracy:** MNIST is a simple, clean, centred dataset, so both networks reach very high accuracy (typically around 98-99% or above). The difference is usually small (a fraction of a percent) and does not justify the extra cost. A high parameter count does not automatically guarantee better test accuracy.
* **Overfitting and generalization:** Compare the train-test gaps printed above. If the training accuracy is clearly higher than test accuracy or the test loss is above the training loss, the model is starting to overfit. AlexNet has far more capacity, but dropout (p = 0.5), a low learning rate and only 3 epochs keep its gap small, while LeNet-5 has little capacity to memorise. Note that the training accuracy printed in the epoch logs is computed with dropout on, so the final train accuracy in the table (measured in eval mode) is a fairer comparison.
* **Conclusion:** For MNIST, LeNet-5 is the better engineering choice (smaller, faster, almost the same accuracy). AlexNet's depth and size pay off on large, complex datasets like ImageNet, not on simple digit classification. Results depend on random initialisation, epochs, optimizer and hardware, so numbers should be taken from your own run.

## Submission Checklist

* [x] Architecture definitions (LeNet-5 and AlexNet-MNIST)
* [x] Shape verification (layer-output tables and the 256 x 6 x 6 check)
* [x] Manual parameter calculations and PyTorch verification
* [x] Training logs and evaluation results
* [x] Completed LeNet-5 vs AlexNet comparison table
* [x] Feature map visualization with interpretation
* [x] Answers to Section 5.1 questions
* [x] Model weights saved: `lenet5_mnist.pth`, `alexnet_mnist.pth`

In [ ]:
# Optional (Colab): download the saved weights to your computer
try:
    from google.colab import files
    files.download("lenet5_mnist.pth")
    files.download("alexnet_mnist.pth")
except Exception as e:
    print("Not running in Colab or download blocked:", e)